## 1. Imports and Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    roc_curve,
    auc,
    precision_recall_curve,
    average_precision_score
)

# Set plotting style
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 10
sns.set_style('whitegrid')

print("✓ Libraries imported successfully")
print()
print("This notebook focuses on METRICS, not model training.")
print("We'll use realistic diagnostic scenarios to understand evaluation metrics.")

---

## 2. The Classification Problem: Healthy vs Faulty

### Recap from Notebook 1

In Notebook 1, you learned that **envelope analysis** reveals bearing faults through peaks in the envelope spectrum at characteristic frequencies:

- **Healthy bearing**: Minimal energy at fault frequencies (BPFI, BPFO, BSF)
- **Inner race fault**: Strong peak at **BPFI** (≈158 Hz at 877 rpm)
- **Outer race fault**: Strong peak at **BPFO** (≈119 Hz at 877 rpm)  
- **Roller fault**: Strong peak at **2×BSF** (≈101 Hz at 877 rpm)

### From Signal Analysis to Classification

A **diagnostic system** analyzes the envelope spectrum and makes a decision:

```
Input: Vibration signal
  ↓
Envelope Analysis (Band-pass + Hilbert)
  ↓
Decision Logic: "Is there a peak at expected fault frequency?"
  ↓
Output: HEALTHY or FAULTY
```

This is a **binary classification problem**:
- **Class 0**: Healthy (No fault detected)
- **Class 1**: Faulty (Fault detected)

### The Challenge: How Good is Our Diagnostic System?

No diagnostic system is perfect. We need metrics to answer:
- How often does it correctly identify faults? (**Recall/Sensitivity**)
- How often are its fault warnings correct? (**Precision**)
- What's the overall accuracy? (**Accuracy**)
- What's the tradeoff between false alarms and missed faults? (**ROC/PR curves**)

---

## 3. Simulated Diagnostic Scenarios

Instead of training models (which requires large datasets), we'll create **realistic scenarios** based on what we learned in Notebook 1.

### Scenario 1: Perfect Diagnostic System

Imagine a diagnostic system that **never makes mistakes**:
- Correctly identifies all 50 healthy bearings as HEALTHY
- Correctly identifies all 50 faulty bearings as FAULTY

Let's see what the metrics look like for this ideal case.

In [ ]:
# Scenario 1: Perfect diagnostic system
n_samples = 100
n_healthy = 50
n_faulty = 50

# Ground truth (actual condition)
y_true = np.array([0]*n_healthy + [1]*n_faulty)  # 0=Healthy, 1=Faulty

# Perfect predictions (system never makes mistakes)
y_pred_perfect = y_true.copy()

print("=== SCENARIO 1: Perfect Diagnostic System ===")
print(f"Total samples: {n_samples}")
print(f"  - Healthy: {n_healthy}")
print(f"  - Faulty: {n_faulty}")
print()
print("Diagnostic Results:")
print(f"  - All healthy bearings correctly identified as HEALTHY")
print(f"  - All faulty bearings correctly identified as FAULTY")

### 3.1. Understanding the Confusion Matrix

A **confusion matrix** shows the four possible outcomes of a binary classifier:

|                    | **Predicted: Healthy** | **Predicted: Faulty** |
|--------------------|------------------------|----------------------|
| **Actual: Healthy** | True Negative (TN)     | False Positive (FP)  |
| **Actual: Faulty**  | False Negative (FN)    | True Positive (TP)   |

**Interpretations:**
- **True Positive (TP)**: System correctly detects a fault → ✅ **Good!**
- **True Negative (TN)**: System correctly identifies healthy bearing → ✅ **Good!**
- **False Positive (FP)**: System raises **false alarm** (healthy diagnosed as faulty) → ⚠️ **Unnecessary maintenance**
- **False Negative (FN)**: System **misses a fault** (faulty diagnosed as healthy) → 🚨 **DANGEROUS! Catastrophic failure risk**

For the perfect system:

In [ ]:
# Compute confusion matrix
cm_perfect = confusion_matrix(y_true, y_pred_perfect)

# Visualize
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm_perfect, annot=True, fmt='d', cmap='Blues', cbar=False,
            xticklabels=['Healthy', 'Faulty'],
            yticklabels=['Healthy', 'Faulty'],
            ax=ax, annot_kws={"size": 16})
ax.set_xlabel('Predicted Condition', fontsize=12, weight='bold')
ax.set_ylabel('Actual Condition', fontsize=12, weight='bold')
ax.set_title('Confusion Matrix - Perfect Diagnostic System', fontsize=14, weight='bold')

plt.tight_layout()
plt.show()

print("Confusion Matrix Breakdown:")
print(f"  True Negatives (TN):  {cm_perfect[0, 0]} - Correctly identified healthy bearings")
print(f"  False Positives (FP): {cm_perfect[0, 1]} - False alarms (healthy called faulty)")
print(f"  False Negatives (FN): {cm_perfect[1, 0]} - Missed faults (faulty called healthy) 🚨")
print(f"  True Positives (TP):  {cm_perfect[1, 1]} - Correctly detected faults")

### 3.2. Classification Metrics

From the confusion matrix, we calculate key performance metrics:

#### **1. Accuracy**
$$\text{Accuracy} = \frac{TP + TN}{TP + TN + FP + FN}$$

**Interpretation**: Overall, what percentage of predictions are correct?  
⚠️ **Problem**: Misleading when classes are imbalanced (e.g., 95 healthy, 5 faulty → 95% accuracy by always predicting healthy!)

#### **2. Precision (Positive Predictive Value)**
$$\text{Precision} = \frac{TP}{TP + FP}$$

**Interpretation**: Of all bearings diagnosed as FAULTY, what percentage actually are faulty?  
**High precision** → Few false alarms → Maintenance teams trust the warnings

#### **3. Recall (Sensitivity, True Positive Rate)**
$$\text{Recall} = \frac{TP}{TP + FN}$$

**Interpretation**: Of all actually faulty bearings, what percentage does the system detect?  
**High recall** → Few missed faults → Critical for safety!

#### **4. F1-Score (Harmonic Mean of Precision and Recall)**
$$F_1 = 2 \times \frac{\text{Precision} \times \text{Recall}}{\text{Precision} + \text{Recall}}$$

**Interpretation**: Balanced metric that considers both false alarms and missed faults.  
Useful when you care equally about precision and recall.

---

Let's calculate these for our perfect system:

In [ ]:
# Calculate metrics for perfect system
accuracy_perfect = accuracy_score(y_true, y_pred_perfect)
precision_perfect = precision_score(y_true, y_pred_perfect)
recall_perfect = recall_score(y_true, y_pred_perfect)
f1_perfect = f1_score(y_true, y_pred_perfect)

print("=== METRICS for Perfect Diagnostic System ===")
print()
print(f"Accuracy:  {accuracy_perfect:.3f} (100%) - All predictions correct")
print(f"Precision: {precision_perfect:.3f} (100%) - No false alarms")
print(f"Recall:    {recall_perfect:.3f} (100%) - No missed faults")
print(f"F1-Score:  {f1_perfect:.3f} (100%) - Perfect balance")
print()
print("✓ This is the ideal case - rarely achievable in practice!")

---

## 4. Scenario 2: Realistic System with Some Errors

Real diagnostic systems make mistakes. Let's simulate a **more realistic scenario** based on what could happen with envelope analysis:

**Assumptions (based on Notebook 1 observations):**
- Envelope analysis is very good at detecting faults (high recall ~95%)
- Sometimes healthy bearings show small peaks that trigger false alarms (precision ~90%)
- The system uses a threshold: "If envelope peak > threshold → FAULTY"

### Why errors occur:
- **False Positives**: Noise or other vibrations create small peaks near fault frequencies
- **False Negatives**: Early-stage faults with very weak signatures might be below threshold

Let's create this scenario:

In [ ]:
# Scenario 2: Realistic system with some errors
np.random.seed(42)  # For reproducibility

# Start with perfect predictions
y_pred_realistic = y_true.copy()

# Introduce realistic errors:
# 1. False Positives: 5 healthy bearings misclassified as faulty (10% false alarm rate)
healthy_indices = np.where(y_true == 0)[0]
fp_indices = np.random.choice(healthy_indices, size=5, replace=False)
y_pred_realistic[fp_indices] = 1

# 2. False Negatives: 2 faulty bearings missed (4% miss rate, ~96% recall)
faulty_indices = np.where(y_true == 1)[0]
fn_indices = np.random.choice(faulty_indices, size=2, replace=False)
y_pred_realistic[fn_indices] = 0

print("=== SCENARIO 2: Realistic Diagnostic System ===")
print()
print("Introduced errors:")
print(f"  - {len(fp_indices)} False Positives (healthy called faulty)")
print(f"  - {len(fn_indices)} False Negatives (faulty called healthy)")
print()
print("This represents a good (but not perfect) diagnostic system.")

In [ ]:
# Confusion matrix for realistic system
cm_realistic = confusion_matrix(y_true, y_pred_realistic)

# Visualize
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm_realistic, annot=True, fmt='d', cmap='Oranges', cbar=False,
            xticklabels=['Healthy', 'Faulty'],
            yticklabels=['Healthy', 'Faulty'],
            ax=ax, annot_kws={"size": 16})
ax.set_xlabel('Predicted Condition', fontsize=12, weight='bold')
ax.set_ylabel('Actual Condition', fontsize=12, weight='bold')
ax.set_title('Confusion Matrix - Realistic Diagnostic System', fontsize=14, weight='bold')

plt.tight_layout()
plt.show()

print("Confusion Matrix Breakdown:")
print(f"  True Negatives (TN):  {cm_realistic[0, 0]} - Correctly identified healthy")
print(f"  False Positives (FP): {cm_realistic[0, 1]} - False alarms ⚠️")
print(f"  False Negatives (FN): {cm_realistic[1, 0]} - Missed faults 🚨")
print(f"  True Positives (TP):  {cm_realistic[1, 1]} - Correctly detected faults")

In [ ]:
# Calculate metrics for realistic system
accuracy_realistic = accuracy_score(y_true, y_pred_realistic)
precision_realistic = precision_score(y_true, y_pred_realistic)
recall_realistic = recall_score(y_true, y_pred_realistic)
f1_realistic = f1_score(y_true, y_pred_realistic)

print("=== METRICS for Realistic Diagnostic System ===")
print()
print(f"Accuracy:  {accuracy_realistic:.3f} ({accuracy_realistic*100:.1f}%) - Still quite good!")
print(f"Precision: {precision_realistic:.3f} ({precision_realistic*100:.1f}%) - About 90% of fault warnings are correct")
print(f"Recall:    {recall_realistic:.3f} ({recall_realistic*100:.1f}%) - Detects 96% of actual faults")
print(f"F1-Score:  {f1_realistic:.3f} ({f1_realistic*100:.1f}%) - Good balance")
print()
print("⚠️ Notice: Accuracy is still high (93%), but we missed 2 faults!")
print("   → This is why we need multiple metrics, not just accuracy.")

---

## 5. Scenario 3: Safety-Critical System (Prioritize Recall)

In bearing diagnostics, **missing a fault is much worse than a false alarm**:
- **False Positive** → Unnecessary inspection → Costs money
- **False Negative** → Catastrophic bearing failure → Equipment damage, safety risk, production stoppage

**Strategy**: Set a **lower threshold** to catch more faults, even if it means more false alarms.

Let's simulate a "conservative" system that prioritizes recall:

In [ ]:
# Scenario 3: Conservative system (high recall, lower precision)
np.random.seed(123)

y_pred_conservative = y_true.copy()

# Many false positives: 15 healthy bearings flagged (30% false alarm rate)
fp_indices_cons = np.random.choice(healthy_indices, size=15, replace=False)
y_pred_conservative[fp_indices_cons] = 1

# Very few false negatives: only 1 fault missed (98% recall)
fn_indices_cons = np.random.choice(faulty_indices, size=1, replace=False)
y_pred_conservative[fn_indices_cons] = 0

print("=== SCENARIO 3: Conservative System (Prioritize Safety) ===")
print()
print("Strategy: Lower threshold → Catch almost all faults, accept more false alarms")
print()
print("Introduced errors:")
print(f"  - {len(fp_indices_cons)} False Positives (many false alarms)")
print(f"  - {len(fn_indices_cons)} False Negative (only 1 fault missed)")
print()
print("This is appropriate for safety-critical applications.")

In [ ]:
# Confusion matrix and metrics for conservative system
cm_conservative = confusion_matrix(y_true, y_pred_conservative)

accuracy_conservative = accuracy_score(y_true, y_pred_conservative)
precision_conservative = precision_score(y_true, y_pred_conservative)
recall_conservative = recall_score(y_true, y_pred_conservative)
f1_conservative = f1_score(y_true, y_pred_conservative)

print("=== METRICS for Conservative System ===")
print()
print(f"Accuracy:  {accuracy_conservative:.3f} ({accuracy_conservative*100:.1f}%) - Lower due to many FPs")
print(f"Precision: {precision_conservative:.3f} ({precision_conservative*100:.1f}%) - Many false alarms")
print(f"Recall:    {recall_conservative:.3f} ({recall_conservative*100:.1f}%) - ✓ Excellent! Catches almost all faults")
print(f"F1-Score:  {f1_conservative:.3f} ({f1_conservative*100:.1f}%) - Lower due to precision/recall tradeoff")
print()
print("💡 Key Insight: High recall (safety) comes at the cost of lower precision (more inspections).")

---

## 6. Comparing All Three Scenarios

Let's visualize how different diagnostic strategies affect the metrics:

In [ ]:
# Create comparison dataframe
comparison_df = pd.DataFrame({
    'System': ['Perfect', 'Realistic', 'Conservative'],
    'Accuracy': [accuracy_perfect, accuracy_realistic, accuracy_conservative],
    'Precision': [precision_perfect, precision_realistic, precision_conservative],
    'Recall': [recall_perfect, recall_realistic, recall_conservative],
    'F1-Score': [f1_perfect, f1_realistic, f1_conservative]
})

print(comparison_df.to_string(index=False))
print()
print("Key Observations:")
print("  • Perfect: Unachievable in practice")
print("  • Realistic: Good balance for most applications")
print("  • Conservative: Best for safety-critical systems (prioritizes recall)")

In [ ]:
# Visualize metric comparison
metrics_to_plot = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
x = np.arange(len(metrics_to_plot))
width = 0.25

fig, ax = plt.subplots(figsize=(12, 6))
ax.bar(x - width, comparison_df.iloc[0, 1:], width, label='Perfect', color='green', alpha=0.7)
ax.bar(x, comparison_df.iloc[1, 1:], width, label='Realistic', color='blue', alpha=0.7)
ax.bar(x + width, comparison_df.iloc[2, 1:], width, label='Conservative', color='orange', alpha=0.7)

ax.set_xlabel('Metric', fontsize=12, weight='bold')
ax.set_ylabel('Score', fontsize=12, weight='bold')
ax.set_title('Comparison of Diagnostic System Performance', fontsize=14, weight='bold')
ax.set_xticks(x)
ax.set_xticklabels(metrics_to_plot)
ax.legend()
ax.set_ylim([0, 1.1])
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

---

## 7. The Problem of Class Imbalance

In real industrial settings, **healthy bearings are much more common than faulty ones**:
- Typical ratio: 95% healthy, 5% faulty (or even more extreme)
- This creates the **class imbalance problem**

### Why Class Imbalance is Dangerous

Let's simulate an extreme case: 95 healthy, 5 faulty bearings

In [ ]:
# Scenario 4: Severe class imbalance (95% healthy, 5% faulty)
n_samples_imb = 100
n_healthy_imb = 95
n_faulty_imb = 5

# Ground truth
y_true_imb = np.array([0]*n_healthy_imb + [1]*n_faulty_imb)

# "Lazy classifier": Always predicts HEALTHY (never raises alarm)
y_pred_lazy = np.zeros(n_samples_imb)  # All zeros = always healthy

print("=== SCENARIO 4: Class Imbalance ===")
print(f"Total samples: {n_samples_imb}")
print(f"  - Healthy: {n_healthy_imb} (95%)")
print(f"  - Faulty: {n_faulty_imb} (5%)")
print()
print("Strategy: 'Lazy classifier' - Always predict HEALTHY")
print("  → Never raises an alarm!")
print("  → What will the metrics look like?")

In [ ]:
# Calculate metrics for lazy classifier
cm_lazy = confusion_matrix(y_true_imb, y_pred_lazy)
accuracy_lazy = accuracy_score(y_true_imb, y_pred_lazy)

print("=== METRICS for 'Lazy Classifier' ===")
print()
print(f"Accuracy: {accuracy_lazy:.3f} ({accuracy_lazy*100:.1f}%) - Looks great!")
print()
print("But wait... let's check the confusion matrix:")
print()
print(cm_lazy)
print()
print(f"  True Negatives:  {cm_lazy[0, 0]} ✓")
print(f"  False Positives: {cm_lazy[0, 1]}")
print(f"  False Negatives: {cm_lazy[1, 0]} 🚨 ALL FAULTS MISSED!")
print(f"  True Positives:  {cm_lazy[1, 1]}")
print()
print("⚠️ DANGER: 95% accuracy, but completely useless!")
print("   → Precision and Recall cannot be calculated (division by zero)")
print("   → This is why accuracy alone is MISLEADING with imbalanced data")

### Key Lesson: Don't Trust Accuracy Alone!

**With class imbalance:**
- ✅ Use **Precision, Recall, F1-Score** instead of accuracy
- ✅ Examine the **confusion matrix** carefully
- ✅ Consider the **cost** of different error types (FP vs FN)

**For bearing diagnostics:**
- False Negatives (missed faults) are catastrophic
- False Positives (false alarms) are costly but acceptable
- → **Prioritize RECALL over precision**

---

## 8. ROC Curves: Visualizing the Tradeoff

The **Receiver Operating Characteristic (ROC)** curve shows how the tradeoff between **True Positive Rate (Recall)** and **False Positive Rate** changes as we adjust the decision threshold.

### Understanding ROC Curves

**Axes:**
- **X-axis**: False Positive Rate (FPR) = FP / (FP + TN) → "False alarm rate"
- **Y-axis**: True Positive Rate (TPR) = TP / (TP + FN) → "Recall"

**Interpretation:**
- **Perfect classifier**: TPR=1, FPR=0 (top-left corner)
- **Random guessing**: Diagonal line (TPR = FPR)
- **Area Under Curve (AUC)**: Overall performance metric
  - AUC = 1.0 → Perfect
  - AUC = 0.5 → Random
  - AUC > 0.9 → Excellent

For diagnostic systems using envelope analysis, we can simulate different thresholds:

In [ ]:
# Simulate "confidence scores" for envelope analysis
# (In reality, these would be the peak amplitudes at fault frequencies)
np.random.seed(42)

# Healthy bearings: low scores (small peaks, mostly noise)
scores_healthy = np.random.beta(2, 5, size=n_healthy) * 0.6

# Faulty bearings: high scores (strong peaks at fault frequencies)
scores_faulty = np.random.beta(5, 2, size=n_faulty) * 0.8 + 0.2

# Combine
y_scores = np.concatenate([scores_healthy, scores_faulty])

# ROC curve
fpr, tpr, thresholds = roc_curve(y_true, y_scores)
roc_auc = auc(fpr, tpr)

print("=== ROC Curve Analysis ===")
print()
print(f"AUC (Area Under Curve): {roc_auc:.3f}")
print()
print("Interpretation:")
if roc_auc > 0.9:
    print("  ✓ Excellent diagnostic performance!")
elif roc_auc > 0.8:
    print("  ✓ Good diagnostic performance")
else:
    print("  ⚠️ Fair performance - room for improvement")

### 8.1. How ROC Curves are Built: Understanding Decision Scores

Before plotting ROC curves, we need to understand **how they are constructed**.

**Key Concept: Decision Scores (Confidence Scores)**

In our envelope analysis example:
- The diagnostic system computes a **score** for each bearing (e.g., peak amplitude at fault frequency)
- Higher score → More likely to be faulty
- We set a **threshold**: if score > threshold → predict FAULTY

**Building the ROC Curve:**
1. For each possible threshold value (from 0 to max score):
   - Count True Positives (TP), False Positives (FP), etc.
   - Calculate TPR = TP/(TP+FN) and FPR = FP/(FP+TN)
   - Plot one point (FPR, TPR)
2. Connect all points → ROC curve

Let's visualize this process:

In [ ]:
# Visualize the distribution of scores for healthy vs faulty bearings
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left plot: Histogram of scores
axes[0].hist(scores_healthy, bins=20, alpha=0.6, label='Healthy', color='blue', edgecolor='black')
axes[0].hist(scores_faulty, bins=20, alpha=0.6, label='Faulty', color='red', edgecolor='black')
axes[0].axvline(x=0.5, color='green', linestyle='--', linewidth=2, label='Example threshold = 0.5')
axes[0].set_xlabel('Decision Score (e.g., envelope peak amplitude)', fontsize=11, weight='bold')
axes[0].set_ylabel('Frequency', fontsize=11, weight='bold')
axes[0].set_title('Score Distributions: Healthy vs Faulty Bearings', fontsize=12, weight='bold')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Right plot: Show some example thresholds and their effects
thresholds_examples = [0.3, 0.5, 0.7]
colors_thresh = ['orange', 'green', 'purple']

for i, thresh in enumerate(thresholds_examples):
    # Predictions at this threshold
    y_pred_thresh = (y_scores >= thresh).astype(int)
    
    # Calculate metrics
    tn = np.sum((y_true == 0) & (y_pred_thresh == 0))
    fp = np.sum((y_true == 0) & (y_pred_thresh == 1))
    fn = np.sum((y_true == 1) & (y_pred_thresh == 0))
    tp = np.sum((y_true == 1) & (y_pred_thresh == 1))
    
    tpr_example = tp / (tp + fn) if (tp + fn) > 0 else 0
    fpr_example = fp / (fp + tn) if (fp + tn) > 0 else 0
    
    axes[1].scatter(fpr_example, tpr_example, s=150, color=colors_thresh[i], 
                   marker='o', edgecolors='black', linewidth=2,
                   label=f'Threshold={thresh:.1f}: TPR={tpr_example:.2f}, FPR={fpr_example:.2f}', 
                   zorder=5)

axes[1].plot([0, 1], [0, 1], 'k--', alpha=0.3, label='Random')
axes[1].set_xlabel('False Positive Rate (FPR)', fontsize=11, weight='bold')
axes[1].set_ylabel('True Positive Rate (TPR)', fontsize=11, weight='bold')
axes[1].set_title('Effect of Different Thresholds', fontsize=12, weight='bold')
axes[1].legend(loc='lower right', fontsize=9)
axes[1].grid(alpha=0.3)
axes[1].set_xlim([-0.05, 1.05])
axes[1].set_ylim([-0.05, 1.05])

plt.tight_layout()
plt.show()

print("💡 Key Observations:")
print()
print("LEFT PLOT - Score Distributions:")
print("  • Healthy bearings: Low scores (blue) - small peaks, mostly noise")
print("  • Faulty bearings: High scores (red) - strong peaks at fault frequencies")
print("  • Overlap region: Where classification errors occur")
print()
print("RIGHT PLOT - Threshold Effect:")
print("  • Low threshold (0.3): High TPR (catches most faults) BUT high FPR (many false alarms)")
print("  • Medium threshold (0.5): Balanced TPR and FPR")
print("  • High threshold (0.7): Low FPR (few false alarms) BUT lower TPR (misses some faults)")
print()
print("→ ROC curve is created by trying ALL possible thresholds and connecting the points!")

### 8.2. The Complete ROC Curve

Now let's generate the **complete ROC curve** by evaluating all possible thresholds:

In [ ]:
# Plot ROC curve
fig, ax = plt.subplots(figsize=(8, 6))

ax.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {roc_auc:.3f})')
ax.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Random classifier')
ax.scatter([0], [1], s=100, color='green', marker='*', zorder=5, label='Perfect classifier')

# Mark some specific operating points
# Conservative: high TPR, accept more FPR
idx_conservative = np.argmin(np.abs(tpr - 0.98))
ax.scatter(fpr[idx_conservative], tpr[idx_conservative], s=100, color='orange', 
           marker='o', zorder=5, label=f'Conservative (Recall≈98%)')

# Balanced
idx_balanced = np.argmin(np.abs((tpr - fpr) - np.max(tpr - fpr)))
ax.scatter(fpr[idx_balanced], tpr[idx_balanced], s=100, color='blue', 
           marker='s', zorder=5, label='Balanced')

ax.set_xlim([-0.05, 1.05])
ax.set_ylim([-0.05, 1.05])
ax.set_xlabel('False Positive Rate', fontsize=12, weight='bold')
ax.set_ylabel('True Positive Rate (Recall)', fontsize=12, weight='bold')
ax.set_title('ROC Curve: Diagnostic System Performance', fontsize=14, weight='bold')
ax.legend(loc="lower right")
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print()
print("💡 Key Insight: Moving along the ROC curve = adjusting the detection threshold")
print("   • Left (low FPR): High threshold → Few alarms, but might miss faults")
print("   • Right (high FPR): Low threshold → Catch more faults, but more false alarms")

---

## 9. Precision-Recall Curves: Better for Imbalanced Data

While ROC curves are widely used, **Precision-Recall (PR) curves** are often better for **imbalanced datasets** (like bearing diagnostics where faults are rare).

**Why PR curves?**
- ROC curves can be overly optimistic when the negative class (healthy) dominates
- PR curves focus on the positive class (faults), which is what we care about

**Axes:**
- **X-axis**: Recall = TP / (TP + FN) → "How many faults did we catch?"
- **Y-axis**: Precision = TP / (TP + FP) → "How many alarms are real faults?"

Let's create a PR curve:

In [ ]:
# Precision-Recall curve
precision_curve, recall_curve, pr_thresholds = precision_recall_curve(y_true, y_scores)
avg_precision = average_precision_score(y_true, y_scores)

# Plot PR curve
fig, ax = plt.subplots(figsize=(8, 6))

ax.plot(recall_curve, precision_curve, color='blue', lw=2, 
        label=f'PR curve (AP = {avg_precision:.3f})')
ax.axhline(y=n_faulty/n_samples, color='red', linestyle='--', lw=2,
           label=f'Baseline (random): {n_faulty/n_samples:.2f}')

ax.set_xlim([-0.05, 1.05])
ax.set_ylim([-0.05, 1.05])
ax.set_xlabel('Recall (Sensitivity)', fontsize=12, weight='bold')
ax.set_ylabel('Precision', fontsize=12, weight='bold')
ax.set_title('Precision-Recall Curve', fontsize=14, weight='bold')
ax.legend(loc="best")
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("=== Precision-Recall Analysis ===")
print()
print(f"Average Precision (AP): {avg_precision:.3f}")
print()
print("💡 Interpreting the curve:")
print("   • Top-right corner: Ideal (high recall AND high precision)")
print("   • Baseline (red line): Performance of random classifier")
print("   • Our curve should be well above baseline for good performance")

---

## 10. Summary and Practical Guidelines

### Key Takeaways

**1. Classification Metrics:**
- **Accuracy**: Overall correctness (misleading with imbalanced data!)
- **Precision**: "How many alarms are real?" (TP / (TP + FP))
- **Recall**: "How many faults did we catch?" (TP / (TP + FN))
- **F1-Score**: Harmonic mean of precision and recall

**2. Confusion Matrix:**
Always examine TP, TN, FP, FN - don't rely on a single metric!

**3. For Bearing Diagnostics:**
- **Prioritize RECALL** → Missing faults is catastrophic
- Accept **lower precision** → False alarms are costly but not catastrophic
- Use **ROC and PR curves** to visualize tradeoffs

**4. Class Imbalance:**
- Very common in predictive maintenance (healthy >> faulty)
- **Don't trust accuracy alone!**
- Use precision, recall, F1, and PR curves

**5. Threshold Selection:**
- **Safety-critical** → Low threshold, high recall (catch all faults)
- **Balanced** → Maximize F1-score
- **Cost-sensitive** → Consider false alarm costs

### Connection to Notebook 1

In Notebook 1, you learned that **envelope analysis** reveals bearing faults through:
- Clear peaks at fault frequencies (BPFI, BPFO, BSF)
- Strong signal-to-noise ratio after band-pass filtering
- Reliable distinction between healthy and faulty bearings

In this notebook, you learned how to **evaluate** such a diagnostic system using proper metrics that account for:
- The critical importance of catching faults (recall)
- The cost of false alarms (precision)
- The reality of imbalanced data (rare faults)

---

### Recommended Approach for Industrial Bearing Monitoring:

1. **Development Phase**:
   - Use envelope analysis (Notebook 1 techniques)
   - Establish baseline thresholds using historical data
   - Aim for **recall > 95%** to minimize missed faults

2. **Deployment Phase**:
   - Monitor precision and recall over time
   - Adjust thresholds based on false alarm rates
   - Track maintenance costs vs. failure prevention

3. **Continuous Improvement**:
   - Collect more labeled data
   - Refine detection algorithms
   - Update thresholds as bearing conditions change

---

**End of Notebook 2**